## The `kubectl` Context

In this notebook, we look at the **context** for `kubectl`.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` tool on your computer.

## View the kubectl configuration file

- kubectl stores information about clusters, contexts and users in its config file.
  - On Linux/Mac, this is stored in the file `~/.kube/config`.
  - On Windows, this is stored in the file `%UserProfile%/.kube/config`.
  - The file contains a setting `current-context` that decides which kubernetes cluster `kubectl` commands are issued against.
- The command `kubectl config view` returns the contents of the configuration file.
  - Notice that the configuration file has the following top-level properties:
    - `apiVersion` contains the version of the API beging used to communicate with the Kubernetes API Server (in this case `v1`).
    - `clusters` is an array containing one `cluster` object for each cluster `kubectl` can communicate with.
    - `contexts` is an array containing one `context` object for each context `kubectl` can use to communicate with a Kubernetes cluster.
    - `current-context` contains the name of the current (active) context, i.e. the context of the Kubernetes cluster `kubectl` commands will be sent to.
    - `kind` is the type of Kubernetes resource this file defines (which is `Config` for a configuration file).
    - `users` is an array containing one `user` object for each Kubernetes user `kubectl` can use to communicate with a Kubernetes cluster.
  - Under the `clusters` array, there is one `cluster` object, with cluster settings, for each cluster `kubectl` can communicate with.
    - Notice that each `cluster` has a `name` (e.g. `minikube`) and connection details in the form of a `certificate-authority` and a `server`.
  - Under the `users` array, there is one `user` object, with user settings, for each cluster user `kubectl` can use to communicate with a Kubernetes cluster.
    - Notice that each `user` has a `name` (e.g. `minikube`) and login details in the form of a `client-certificate` and a `client-key`.
  - Under the `contexts` array, there is one `context` object, with context settings, for each context `kubectl` can use to communicate with a Kubernetes cluster.
    - Notice that each `context` has a `name` (e.g. `minikube`), the name of a `cluster` object, the name of a `user` object, and a `namespace`.
      - The name of the `cluster` object refers to a named `cluster` under the `clusters` array.
      - The name of the `user` object refers to a named `user` under the `users` array.
      - The `namespace` contains the name of the namespace `kubectl` commands will target by default.
      - The context `name` is used by `kubectl` to refer to this context that groups together a `cluster` object, a `user` object, and the default `namespace`.
- Notice that:
  - The `current-context` is set to `minikube`.
  - This refers to the `context` (under the `contexts` array) with `name = minikube`.
  - The `context` with `name = minikube` has a `cluster = minikube`, a `user = minikube`, and a `namespace = default`.
    - This means that this context is refering to the `minikube` cluster object, the `minikube` user object, and uses namepace `default` by default.
    - So any `kubectl` commands issued will be send to the `minikube` cluster as the `minikube` user, and will target namespace `default` by default.

In [25]:
!kubectl config view

apiVersion: v1
clusters:
- cluster:
    certificate-authority: /home/student/.minikube/ca.crt
    extensions:
    - extension:
        last-update: Sat, 14 Mar 2026 16:26:33 CET
        provider: minikube.sigs.k8s.io
        version: v1.38.1
      name: cluster_info
    server: https://192.168.49.2:8443
  name: minikube
contexts:
- context:
    cluster: minikube
    extensions:
    - extension:
        last-update: Sat, 14 Mar 2026 16:26:33 CET
        provider: minikube.sigs.k8s.io
        version: v1.38.1
      name: context_info
    namespace: default
    user: minikube
  name: minikube
current-context: minikube
kind: Config
users:
- name: minikube
  user:
    client-certificate: /home/student/.minikube/profiles/minikube/client.crt
    client-key: /home/student/.minikube/profiles/minikube/client.key


## Get current context

- kubectl stores information about clusters, contexts and users in its config file.
  - On Linux/Mac, this is stored in the file `~/.kube/config`.
  - On Windows, this is stored in the file `%UserProfile%/.kube/config`.
  - The file contains a setting `current-context` that decides which kubernetes cluster `kubectl` commands are issued against.
- The command `kubectl config current-context` returns the current context `kubectl` is using.
  - If you are using a default Minikube Kubernetes cluster, the context is `minikube`, i.e. `kubectl` commands are run against the `minikube` cluster.

In [1]:
!kubectl config current-context

minikube


## List all contexts

- kubectl's config file can store information about multiple kubernetes clusters
  - For example, a Docker Desktop cluster and an Azure cluster.
- We can list all contexts with the command `kubectl config get-contexts`.
  - The list of all contexts on your computer will differ depending on the clusters you have configured on your computer.

For example:
- In this case, `minikube` is a Minikube cluster.
- The `*` shows us the `current context`, i.e. the cluster that `kubectl` commands are run against.

In [2]:
!kubectl config get-contexts

CURRENT   NAME       CLUSTER    AUTHINFO   NAMESPACE
*         minikube   minikube   minikube   default


Let's start another single-node Minikube cluster called `minikube2`:

In [7]:
!minikube start -p minikube2

😄  [minikube2] minikube v1.38.1 on Ubuntu 24.04 (kvm/amd64)
✨  Automatically selected the docker driver. Other choices: kvm2, ssh
❗  Starting v1.39.0, minikube will default to "containerd" container runtime. See #21973 for more info.
📌  Using Docker driver with root privileges
👍  Starting "minikube2" primary control-plane node in "minikube2" cluster
🚜  Pulling base image v0.0.50 ...
🔥  Creating docker container (CPUs=2, Memory=3900MB) ...7m\
🐳  Preparing Kubernetes v1.35.1 on Docker 29.2.1 ...7m/
🔗  Configuring bridge CNI (Container Networking Interface) ...?25h
🔎  Verifying Kubernetes components...
    ▪ Using image gcr.io/k8s-minikube/storage-provisioner:v5
🌟  Enabled addons: storage-provisioner, default-storageclass
🏄  Done! kubectl is now configured to use "minikube2" cluster and "default" namespace by default


Now, let's list the contexts again:
- In this case, `minikube` and `minikube2` are both Minikube clusters.
- We see that the `*` is now next to `minikube2`, which is now the current (active) context, i.e. the cluster that `kubectl` commands are run against.

In [9]:
!kubectl config get-contexts

CURRENT   NAME        CLUSTER     AUTHINFO    NAMESPACE
          minikube    minikube    minikube    default
*         minikube2   minikube2   minikube2   default


## Change context

- We can change context using the command `kubectl config use-context [ContextName]`
  - `[contextname]` is the name of the context we want to change to.
- Let's swich to the `minikube` context.

In [10]:
!kubectl config use-context minikube

Switched to context "minikube".


- Now we see that `minikube` is the current context, i.e. the cluster that `kubectl` commands are run against.

In [11]:
!kubectl config get-contexts

CURRENT   NAME        CLUSTER     AUTHINFO    NAMESPACE
*         minikube    minikube    minikube    default
          minikube2   minikube2   minikube2   default


## Rename context

A context can be renamed with the command `kubectl config rename-context [OldContextName] [NewContextName]`.
- Let's rename `minikube2` to `minikube3`.

In [12]:
!kubectl config rename-context minikube2 minikube3

Context "minikube2" renamed to "minikube3".


- We see that the name of the context has changed to `minikube3`, but it is still referring to the `minikube2` Kubernetes cluster.
  - We have only renamed the context, i.e. the name we use to refer to the cluster.

In [14]:
!kubectl config get-contexts

CURRENT   NAME        CLUSTER     AUTHINFO    NAMESPACE
*         minikube    minikube    minikube    default
          minikube3   minikube2   minikube2   default


## Delete context

- A context can be deleted from the config file with the command `kubectl config delete-context [contextName]`.
- Let's delete the `minikube3` context.

In [15]:
!kubectl config delete-context minikube3

deleted context minikube3 from /home/student/.kube/config


- Now we have no way to refer to the `minikube2` cluster.

In [20]:
!kubectl config get-contexts

CURRENT   NAME       CLUSTER    AUTHINFO   NAMESPACE
*         minikube   minikube   minikube   default


- Although, we can still list the Minikube clusters via the `minikube` CLI.

In [21]:
!minikube profile list

┌───────────┬────────┬─────────┬──────────────┬─────────┬────────┬───────┬────────────────┬────────────────────┐
│  PROFILE  │ DRIVER │ RUNTIME │      IP      │ VERSION │ STATUS │ NODES │ ACTIVE PROFILE │ ACTIVE KUBECONTEXT │
├───────────┼────────┼─────────┼──────────────┼─────────┼────────┼───────┼────────────────┼────────────────────┤
│ minikube  │ docker │ docker  │ 192.168.49.2 │ v1.35.1 │ OK     │ 1     │ *              │ *                  │
│ minikube2 │ docker │ docker  │ 192.168.58.2 │ v1.35.1 │ OK     │ 1     │                │                    │
└───────────┴────────┴─────────┴──────────────┴─────────┴────────┴───────┴────────────────┴────────────────────┘


- Let's stop and delete the `minikube2` cluster.

In [22]:
!minikube stop -p minikube2
!minikube delete -p minikube2

✋  Stopping node "minikube2"  ...
🛑  Powering off "minikube2" via SSH ...
🛑  1 node stopped.
🔥  Deleting "minikube2" in docker ...
🔥  Deleting container "minikube2" ...
🔥  Removing /home/student/.minikube/machines/minikube2 ...
💀  Removed all traces of the "minikube2" cluster.
